# COVID-19 Data Pipeline & Dashboard — Part 2: Interactive Dashboard

An interactive **Dash** app over the cleaned dataset from [`01_data_preprocessing.ipynb`](01_data_preprocessing.ipynb): country dropdown, date-range picker, and 4 linked time-series charts (confirmed, deaths, recovered, active).

**Run this after** `01_data_preprocessing.ipynb` has produced `Covid_19_clean_complete_final.csv`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdelrhmanhesham1/applied-ml-notebooks/blob/main/projects/covid19-data-pipeline-dashboard/02_interactive_dashboard.ipynb)


## 1. Setup & Load

In [ ]:
# Install necessary libraries
!pip install dash jupyter-dash pandas plotly




In [2]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
import matplotlib.pyplot as plt

# Load your preprocessed dataset
df = pd.read_csv("/content/Covid_19_clean_complete_final.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24590 entries, 0 to 24589
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Province/State  24590 non-null  object 
 1   Country/Region  24590 non-null  object 
 2   Lat             24590 non-null  float64
 3   Long            24590 non-null  float64
 4   Date            24590 non-null  object 
 5   Confirmed       24590 non-null  int64  
 6   Deaths          24590 non-null  int64  
 7   Recovered       24590 non-null  int64  
 8   Active          24590 non-null  int64  
 9   WHO Region      24590 non-null  object 
dtypes: float64(2), int64(4), object(4)
memory usage: 1.9+ MB


## 2. Quick Static Look

Before building the interactive app, a fast sanity-check: global confirmed-cases trend and a correlation matrix across the numeric columns.


In [ ]:
# Convert data types
df['Date'] = pd.to_datetime(df['Date'])
df['Confirmed'] = df['Confirmed'].astype(int)
df['Deaths'] = df['Deaths'].astype(int)
df['Recovered'] = df['Recovered'].astype(int)
df['Active'] = df['Confirmed'] - df['Deaths'] - df['Recovered']
# Example: Time series plot of confirmed cases
plt.figure(figsize=(12, 6))
sns.lineplot(data=df, x='Date', y='Confirmed')
plt.title('Time Series of Confirmed Cases')
plt.show()

# Group by country and sum confirmed cases
x = df.groupby('Country/Region')['Confirmed'].sum().reset_index()
print(x)
# Select only numeric columns for the correlation matrix
numeric_columns = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_columns].corr()
# Example: Correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()



![Confirmed cases time series](screenshots/confirmed-cases-timeseries.png)

![Correlation matrix](screenshots/correlation-matrix.png)

## 3. Interactive Dashboard

Builds a `JupyterDash` app with a country dropdown (defaults to `World`, summed across all countries), a date-range picker, and 4 linked line charts.

In [ ]:
# Import libraries
import pandas as pd
import plotly.express as px
from jupyter_dash import JupyterDash
from dash import dcc, html
from dash.dependencies import Input, Output

# Initialize Dash app
app = JupyterDash(__name__)
# Layout of the dashboard
app.layout = html.Div([
    html.H1('COVID-19 Dashboard', style={'textAlign': 'center'}),
# Dropdown to select a country
    dcc.Dropdown(
        id='country-dropdown',
        options=[{'label': 'World', 'value': 'World'}] +
                [{'label': country, 'value': country} for country in
df['Country/Region'].unique()],
        value='World', # Default value
        placeholder='Select a Country',
        style={'width': '50%', 'margin': 'auto'}
    ),
# Date picker range
    dcc.DatePickerRange(
        id='date-picker-range',
        start_date=df['Date'].min(),
        end_date=df['Date'].max(),
        display_format='YYYY-MM-DD',
        style={'textAlign': 'center', 'margin': 'auto'}
    ),
# Graphs for each metric
    dcc.Graph(id='confirmed-cases'),
    dcc.Graph(id='deaths-cases'),
    dcc.Graph(id='recovered-cases'),
    dcc.Graph(id='active-cases')
])
# Callback to update graphs
@app.callback(
    [Output('confirmed-cases', 'figure'),
    Output('deaths-cases', 'figure'),
    Output('recovered-cases', 'figure'),
    Output('active-cases', 'figure')],
    [Input('country-dropdown', 'value'),
    Input('date-picker-range', 'start_date'),
    Input('date-picker-range', 'end_date')]
)
def update_graphs(selected_country, start_date, end_date):
# Filter data based on date range
  filtered_df = df[(df['Date'] >= start_date) &
(df['Date'] <= end_date)]

# Handle "World" case by summing all values

  if selected_country == 'World':
        filtered_df = filtered_df.groupby('Date').sum().reset_index()
  else:
        filtered_df = filtered_df[filtered_df['Country/Region'] ==
selected_country]

# Debugging filtered data
  print(f"Filtered data shape: {filtered_df.shape}")
  print(filtered_df.head())
# Create line charts for each metric
  confirmed_fig = px.line(
      filtered_df, x='Date', y='Confirmed',
      title=f'Confirmed Cases Over Time in {selected_country}'
  )
  deaths_fig = px.line(
      filtered_df, x='Date', y='Deaths',
      title=f'Deaths Over Time in {selected_country}'
  )
  recovered_fig = px.line(
      filtered_df, x='Date', y='Recovered',
      title=f'Recovered Cases Over Time in {selected_country}'
  )
  active_fig = px.line(
      filtered_df, x='Date', y='Active',
      title=f'Active Cases Over Time in {selected_country}'
  )
  return confirmed_fig, deaths_fig, recovered_fig, active_fig


## 4. Run

In Colab/Jupyter, `mode='inline'` renders the dashboard directly below this cell. Running locally outside a notebook, use `app.run(debug=True)` instead and open the printed `http://127.0.0.1:8050` URL.

In [5]:
# Run the app
app.run(mode='inline', debug=True, port=8050)

<IPython.core.display.Javascript object>

## Future Improvements

- Add a log-scale toggle for the case-count charts (raw counts compress early-pandemic detail)
- Cache the country groupby instead of recomputing it on every callback
- Deploy the Dash app (Render/Railway free tier) instead of only running inline in a notebook
